In [2]:
# INSTALL PYSPARK
!pip install pyspark

In [3]:
# CREATE SPARK SESSION

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("CustomerOrderETL").getOrCreate()

In [4]:
# UPLOAD CSV FILE
from google.colab import files
uploaded = files.upload()

# Upload week4_cleaned_orders.csv

Saving week4_cleaned_orders.csv to week4_cleaned_orders (1).csv


In [5]:
#READ AND PRINT THE ORIGINAL DATA

df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("week4_cleaned_orders.csv")
print("ORIGINAL DATA")
df.show()

ORIGINAL DATA
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+
|order_id|customer_id|customer_name|region|product_name|order_date|delivery_date|delivery_days|delayed|   status|
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+
|     101|          1|        Selva| South|      Laptop|2026-05-01|   2026-05-10|            9|      1|Delivered|
|     102|          1|        Selva| South|       Mouse|2026-05-03|   2026-05-05|            2|      0|Delivered|
|     103|          2|         Arun| North|    Keyboard|2026-05-02|   2026-05-12|           10|      1|  Delayed|
|     104|          3|        Priya|  West|      Mobile|2026-05-04|   2026-05-06|            2|      0|Delivered|
|     105|          4|        Divya|  East|      Tablet|2026-05-05|   2026-05-15|           10|      1|  Delayed|
|     106|          5|        Kumar| South|     Monitor|2026-05-06|   2026

In [6]:
from pyspark.sql.functions import when, col

# CREATE delayed_flag COLUMN

df = df.withColumn("delayed_flag",when(col("delivery_days") > 5, "YES").otherwise("NO"))

# SHOW TRANSFORMED DATA
print("TRANSFORMED DATA")
df.show()

TRANSFORMED DATA
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+------------+
|order_id|customer_id|customer_name|region|product_name|order_date|delivery_date|delivery_days|delayed|   status|delayed_flag|
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+------------+
|     101|          1|        Selva| South|      Laptop|2026-05-01|   2026-05-10|            9|      1|Delivered|         YES|
|     102|          1|        Selva| South|       Mouse|2026-05-03|   2026-05-05|            2|      0|Delivered|          NO|
|     103|          2|         Arun| North|    Keyboard|2026-05-02|   2026-05-12|           10|      1|  Delayed|         YES|
|     104|          3|        Priya|  West|      Mobile|2026-05-04|   2026-05-06|            2|      0|Delivered|          NO|
|     105|          4|        Divya|  East|      Tablet|2026-05-05|   2026-05-15|           10

In [7]:
# FILTER DELAYED ORDERS
delayed_df = df.filter(col("delayed_flag")=="YES")
print("DELAYED ORDERS")
delayed_df.show()


DELAYED ORDERS
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+------------+
|order_id|customer_id|customer_name|region|product_name|order_date|delivery_date|delivery_days|delayed|   status|delayed_flag|
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+------------+
|     101|          1|        Selva| South|      Laptop|2026-05-01|   2026-05-10|            9|      1|Delivered|         YES|
|     103|          2|         Arun| North|    Keyboard|2026-05-02|   2026-05-12|           10|      1|  Delayed|         YES|
|     105|          4|        Divya|  East|      Tablet|2026-05-05|   2026-05-15|           10|      1|  Delayed|         YES|
|     107|          6|         Ravi| North|     Printer|2026-05-07|   2026-05-20|           13|      1|  Delayed|         YES|
|     109|          8|        Vijay|  East|      Camera|2026-05-09|   2026-05-18|            9| 

In [8]:
# SAVE AS CSV
df.write.format("csv").option("header",True).mode("overwrite").save("customer_orders_csv")
print("CSV OUTPUT SAVED SUCCESSFULLY")

CSV OUTPUT SAVED SUCCESSFULLY


In [9]:
# CREATE TEMP VIEW
df.createOrReplaceTempView("customer_orders")

In [10]:
# TOP 5 DELAYED CUSTOMERS

top_delayed_customers = spark.sql("""

SELECT
    customer_name,
    COUNT(*) AS total_delays
FROM customer_orders
WHERE delayed = 1
GROUP BY customer_name
ORDER BY total_delays DESC
LIMIT 5

""")



print("TOP 5 DELAYED CUSTOMERS")
top_delayed_customers.show()

TOP 5 DELAYED CUSTOMERS
+-------------+------------+
|customer_name|total_delays|
+-------------+------------+
|         Arun|           1|
|        Divya|           1|
|         Ravi|           1|
|        Vijay|           1|
|        Selva|           1|
+-------------+------------+



In [11]:
# SHOW FINAL DATA
print("FINAL DATA")
df.show()

# STOP SPARK SESSION
spark.stop()

FINAL DATA
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+------------+
|order_id|customer_id|customer_name|region|product_name|order_date|delivery_date|delivery_days|delayed|   status|delayed_flag|
+--------+-----------+-------------+------+------------+----------+-------------+-------------+-------+---------+------------+
|     101|          1|        Selva| South|      Laptop|2026-05-01|   2026-05-10|            9|      1|Delivered|         YES|
|     102|          1|        Selva| South|       Mouse|2026-05-03|   2026-05-05|            2|      0|Delivered|          NO|
|     103|          2|         Arun| North|    Keyboard|2026-05-02|   2026-05-12|           10|      1|  Delayed|         YES|
|     104|          3|        Priya|  West|      Mobile|2026-05-04|   2026-05-06|            2|      0|Delivered|          NO|
|     105|          4|        Divya|  East|      Tablet|2026-05-05|   2026-05-15|           10|     